<a href="https://colab.research.google.com/github/Aiza368/New-folder/blob/main/Project_03_LangChain_Function_Tool_Calling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install -q -U langchain-google-genai
!pip install yt-dlp
!pip install -q openai-whisper
from langchain.agents import ConversationalAgent
from langchain_google_genai import GoogleGenerativeAI
import yt_dlp
import whisper
from transformers import pipeline
import nltk
from nltk.tokenize import sent_tokenize
nltk.download('punkt')

# Define the download_audio function
def download_audio(video_url):
  try:
    ydl_opts = {
        'format': 'bestaudio',
        'outtmpl': 'audio.%(ext)s',
        'noplaylist': True
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info_dict = ydl.extract_info(video_url, download=True)
        audio_file = ydl.prepare_filename(info_dict)
        return audio_file
  except Exception as e:
    return f"Error downloading audio: {e}"

# Define the transcribe_audio function if not already defined
def transcribe_audio(audio_file):
  try:
    model = whisper.load_model("base")
    result = model.transcribe(audio_file)
    return result["text"]
  except Exception as e:
    return f"Error transcribing audio: {e}"

# Define the summarize_text function if not already defined
def summarize_text(text):
  try:
    summarizer = pipeline("summarization")
    summary = summarizer(text, max_length=130, min_length=30, do_sample=False)
    return summary[0]["summary_text"]
  except Exception as e:
    return f"Error summarizing text: {e}"

# ... (rest of your code)
def summarize_video_external(video_url):
    audio_file = download_audio(video_url)
    if "Error" in audio_file:
        return audio_file
    transcription = transcribe_audio(audio_file)
    if "Error" in transcription:
        return transcription
    summary = summarize_text(transcription)
    import os
    try:
        os.remove(audio_file)
    except:
        pass
    return summary

# Now, to use this function in LangChain:
from langchain.agents import Tool
tools = [
    Tool(
        name="summarize_video",
        func=summarize_video_external,
        description="Useful for summarizing youtube videos. Input should be a youtube url."
    )
]

# Get your Google API key (if required)
from google.colab import userdata
GOOGLE_API_KEY= userdata.get('GOOGLE_API_KEY_1')

llm = GoogleGenerativeAI(
    model="gemini-2.0-flash-exp",
    api_key=GOOGLE_API_KEY  # Set the API key here if required
)
from langchain.chains import LLMChain
from langchain.agents import AgentType, initialize_agent, Tool
# Create an LLMChain
llm_chain = LLMChain(llm=llm, prompt=ConversationalAgent.create_prompt(tools))

# Initialize the agent with the llm_chain
agent = initialize_agent(tools, llm, agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION, verbose=True, llm_chain=llm_chain,)

chat_history = []


# Test the Custom Tools
user_input = "Can you summarize this video for me? [https://www.youtube.com/watch?v=XZrckLYqdys]"
print(f"User: {user_input}")
result = agent.run({"input": user_input, "chat_history": chat_history})  # Pass chat_history
print(result)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


User: Can you summarize this video for me? [https://www.youtube.com/watch?v=XZrckLYqdys]


> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? Yes
Action: summarize_video
Action Input: https://www.youtube.com/watch?v=XZrckLYqdys
```
[generic] Extracting URL: https://www.youtube.com/watch?v=XZrckLYqdys
```

[generic] watch?v=XZrckLYqdys
```
: Downloading webpage
[redirect] Following redirect to https://www.youtube.com/watch?v=XZrckLYqdys%60%60%60
[youtube] Extracting URL: https://www.youtube.com/watch?v=XZrckLYqdys%60%60%60
[youtube] XZrckLYqdys: Downloading webpage
[youtube] XZrckLYqdys: Downloading tv player API JSON
[youtube] XZrckLYqdys: Downloading ios player API JSON
[youtube] XZrckLYqdys: Downloading m3u8 information
[info] XZrckLYqdys: Downloading 1 format(s): 251
[download] Destination: audio.webm
[download] 100% of    4.19MiB in 00:00:00 at 19.76MiB/s  


/usr/local/lib/python3.11/dist-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(fp, map_location=device)
/usr/local/li


Observation:  Information technology is the use of computer systems, storages, networks and tangible devices, technological infrastructure, systems and processes to create, modify, transfer, store, multiply, and exchange electronic data . Data is every type of information from facts, statistics, numerical sequences, and patterns which is analyzed, studied, compared, modified, stored, and overall used as a component . The value of information technology is that although people are capable of handling data using our brains and even the old paper and pen processes, these processes are not as reliable, efficient, and effective as relying on information technology .
Thought:Do I need to use a tool? No
AI: The video discusses information technology and its importance. It explains that information technology involves using computer systems, storages, networks, and other technological infrastructure to create, modify, transfer, store, and exchange electronic data. The video also defines data 

# ***MARKDOWN FORMAT***



---

# YouTube Video Summarizer Using LangChain and External Tools 🎥🤖


---

## 🚀 Installing the Required Libraries

```python
!pip install -q -U langchain-google-genai
!pip install yt-dlp
!pip install -q openai-whisper
```

### 📌 **What It Does**:
- **`langchain-google-genai`**: Allows us to use Google's Gemini AI models in our code.
- **`yt-dlp`**: Helps download audio from YouTube videos.
- **`openai-whisper`**: Enables transcription of audio into text.

---

## 📦 Importing Required Modules

```python
from langchain.agents import ConversationalAgent
from langchain_google_genai import GoogleGenerativeAI
import yt_dlp
import whisper
from transformers import pipeline
import nltk
from nltk.tokenize import sent_tokenize
nltk.download('punkt')
```

### 📌 **What It Does**:
- **LangChain Modules**: Used for building and running AI agents.
- **`yt_dlp`**: Downloads audio from YouTube videos.
- **`whisper`**: Transcribes audio into text.
- **`pipeline`**: Summarizes text.
- **`nltk`**: Tokenizes text into sentences.

---

## 🎧 Defining the `download_audio` Function

```python
def download_audio(video_url):
    try:
        ydl_opts = {
            'format': 'bestaudio',
            'outtmpl': 'audio.%(ext)s',
            'noplaylist': True
        }
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info_dict = ydl.extract_info(video_url, download=True)
            audio_file = ydl.prepare_filename(info_dict)
            return audio_file
    except Exception as e:
        return f"Error downloading audio: {e}"
```

### 💡 **Explanation**:
- **🎯 Purpose**: Downloads audio from a YouTube video.
- **⚙️ How It Works**:
  - Sets options like downloading the best audio format and ignoring playlists.
  - Uses `yt_dlp` to download and save the audio file.
  - Returns the file name or an error if something goes wrong.

---

## 📝 Defining the `transcribe_audio` Function

```python
def transcribe_audio(audio_file):
    try:
        model = whisper.load_model("base")
        result = model.transcribe(audio_file)
        return result["text"]
    except Exception as e:
        return f"Error transcribing audio: {e}"
```

### 💡 **Explanation**:
- **🎯 Purpose**: Converts audio into text using OpenAI's Whisper model.
- **⚙️ How It Works**:
  - Loads the Whisper `base` model.
  - Transcribes the audio file into text.
  - Returns the transcription or an error.

---

## ✂️ Defining the `summarize_text` Function

```python
def summarize_text(text):
    try:
        summarizer = pipeline("summarization")
        summary = summarizer(text, max_length=130, min_length=30, do_sample=False)
        return summary[0]["summary_text"]
    except Exception as e:
        return f"Error summarizing text: {e}"
```

### 💡 **Explanation**:
- **🎯 Purpose**: Summarizes long text into a concise format.
- **⚙️ How It Works**:
  - Uses Hugging Face's `pipeline` for summarization.
  - Configures maximum and minimum lengths for the summary.
  - Returns the summarized text or an error message.

---

## 🔗 Combining Everything: `summarize_video_external` Function

```python
def summarize_video_external(video_url):
    audio_file = download_audio(video_url)
    if "Error" in audio_file:
        return audio_file
    transcription = transcribe_audio(audio_file)
    if "Error" in transcription:
        return transcription
    summary = summarize_text(transcription)
    import os
    try:
        os.remove(audio_file)
    except:
        pass
    return summary
```

### 💡 **Explanation**:
- **🎯 Purpose**: Combines all steps to summarize a YouTube video.
- **⚙️ How It Works**:
  1. Downloads the video's audio.
  2. Transcribes the audio into text.
  3. Summarizes the transcribed text.
  4. Deletes the temporary audio file.
  5. Returns the summary or an error if any step fails.

---

## 🛠️ Creating Tools for LangChain

```python
from langchain.agents import Tool

tools = [
    Tool(
        name="summarize_video",
        func=summarize_video_external,
        description="Useful for summarizing YouTube videos. Input should be a YouTube URL."
    )
]
```

### 💡 **Explanation**:
- **🎯 Purpose**: Defines a LangChain tool for summarizing YouTube videos.
- **⚙️ How It Works**:
  - Creates a `Tool` object linking the `summarize_video_external` function with a descriptive name and instructions.

---

## 🔑 Setting Up Google Generative AI

```python
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY_1')

llm = GoogleGenerativeAI(
    model="gemini-2.0-flash-exp",
    api_key=GOOGLE_API_KEY
)
```

### 💡 **Explanation**:
- **🎯 Purpose**: Configures Google's Gemini AI model.
- **⚙️ How It Works**:
  - Retrieves the API key from Colab's user data.
  - Uses the `gemini-2.0-flash-exp` model for advanced responses.

---

## 🤖 Initializing LangChain Agent

```python
from langchain.chains import LLMChain
from langchain.agents import AgentType, initialize_agent

llm_chain = LLMChain(llm=llm, prompt=ConversationalAgent.create_prompt(tools))

agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    llm_chain=llm_chain,
)
```

### 💡 **Explanation**:
- **🎯 Purpose**: Sets up a conversational agent using LangChain.
- **⚙️ How It Works**:
  - Links the tools and the LLMChain to the agent.
  - Configures it to handle conversational inputs.

---

## 🔍 Testing the Agent

```python
chat_history = []

user_input = "Can you summarize this video for me? [https://www.youtube.com/watch?v=XZrckLYqdys]"
print(f"User: {user_input}")
result = agent.run({"input": user_input, "chat_history": chat_history})
print(result)
```

### 💡 **Explanation**:
- **🎯 Purpose**: Tests the summarizer by inputting a YouTube video URL.
- **⚙️ How It Works**:
  - Simulates a user query for summarizing a YouTube video.
  - Prints the agent's response.













